# Lesson 2: File Handling & I/O

**Week 3 · Data Engineering Course**

---

File I/O is where most data engineering pipelines begin and end. This lesson explains what actually happens when Python opens a file, why encoding errors occur and how to prevent them, how the context manager protocol guarantees cleanup, and how to process files that are too large to fit in memory.

**Sections:**
1. Files at the OS Level
2. Encoding in Depth
3. The Context Manager Protocol
4. Streaming Large Files
5. Exception Handling for I/O
6. pathlib in Depth

In [ ]:
import csv
import io
import os
import sys
import tracemalloc
from pathlib import Path
from contextlib import contextmanager

RAW = Path('../data/raw')
print('Setup complete.')

---

## 2.1 Files at the OS Level

When Python opens a file, it asks the operating system to create a **file descriptor** — an integer that the OS uses to track the open file. Python wraps this in a file object and adds buffering and encoding layers on top.

Three layers sit between your Python code and the actual bytes on disk:

1. **Python file object** — the `f` you use in your code
2. **Python I/O buffer** — batches reads/writes into chunks (default 8 KB) to avoid a system call per byte
3. **OS kernel buffer** — the OS's own cache; reading from here is faster than hitting the disk

Each open file consumes one file descriptor. Operating systems have limits on how many a process can hold. `ulimit -n` shows the limit on Linux/Mac. The `with` statement exists partly to guarantee file descriptors are released.

In [ ]:
# Inspect a file object's internals after opening
path = RAW / 'sales_q1.csv'

with open(path, 'r', encoding='utf-8') as f:
    print(f'type:         {type(f)}')
    print(f'name:         {f.name}')
    print(f'mode:         {f.mode}')
    print(f'encoding:     {f.encoding}')
    print(f'buffer:       {f.buffer}')         # BufferedReader wrapping the raw FileIO
    print(f'readable:     {f.readable()}')
    print(f'closed:       {f.closed}')         # False — file is open
    fd = f.fileno()                            # the OS-level file descriptor integer
    print(f'file descr:   {fd}')

print(f'closed after with: {f.closed}')  # True — with block closed it automatically

In [ ]:
# Text mode vs binary mode
# In TEXT mode (mode='r'), Python:
#   - decodes bytes to str using the specified encoding
#   - translates \r\n (Windows line endings) to \n on all platforms
# In BINARY mode (mode='rb'), Python gives you raw bytes with no translation

with open(path, 'rb') as f:       # binary mode
    first_bytes = f.read(50)
    print(f'binary: {first_bytes}')

with open(path, 'r', encoding='utf-8') as f:  # text mode
    first_chars = f.read(50)
    print(f'text:   {repr(first_chars)}')

# Binary mode is needed when:
# - detecting encoding (you must read raw bytes first)
# - reading non-text files (images, Parquet, Avro)
# - checking for a BOM (byte order mark) at the start of a file

In [ ]:
# Buffering: Python does not read one byte at a time — it reads in chunks
# You can see the buffer size:
with open(path, 'r', encoding='utf-8') as f:
    print(f'default buffer size: {f.buffer.raw.DEFAULT_BUFFER_SIZE} bytes')
    # Typically 8192 bytes = 8 KB

# You can control buffer size:
with open(path, 'r', encoding='utf-8', buffering=65536) as f:
    # 64 KB buffer — reads from disk less frequently, useful for large sequential reads
    content = f.read()
    print(f'read {len(content)} characters with 64 KB buffer')

---

## 2.2 Encoding in Depth

**Encoding** is the mapping between bytes and characters. Different encodings use different mappings.

- **ASCII**: 128 characters, 1 byte each. No accents, no non-Latin scripts.
- **Latin-1 (ISO-8859-1)**: 256 characters, 1 byte each. Adds Western European characters. Every byte is valid — it never raises a decode error, which makes it dangerously silent.
- **cp1252 (Windows-1252)**: Microsoft's extension of Latin-1. Common for files created on Windows in Western Europe.
- **UTF-8**: Variable length (1–4 bytes per character). Backwards-compatible with ASCII. The universal standard. Always prefer UTF-8 when you control the encoding.

The most common error in data engineering: a file was written in cp1252 or Latin-1 and you read it as UTF-8. Certain byte sequences that are valid in those encodings are invalid in UTF-8.

In [ ]:
# Demonstrate encoding differences
text = 'Caf\u00e9 au lait'  # Café au lait — contains a non-ASCII character

utf8_bytes   = text.encode('utf-8')
latin1_bytes = text.encode('latin-1')
cp1252_bytes = text.encode('cp1252')

print(f'Original:  {text}')
print(f'UTF-8:     {utf8_bytes}    ({len(utf8_bytes)} bytes)')
print(f'Latin-1:   {latin1_bytes}  ({len(latin1_bytes)} bytes)')
print(f'cp1252:    {cp1252_bytes}  ({len(cp1252_bytes)} bytes)')
print()

# Reading Latin-1 bytes as UTF-8 fails
try:
    latin1_bytes.decode('utf-8')
except UnicodeDecodeError as e:
    print(f'Error reading latin-1 bytes as utf-8: {e}')

# Reading UTF-8 bytes as Latin-1 succeeds but gives WRONG output
bad_decode = utf8_bytes.decode('latin-1')
print(f'UTF-8 bytes decoded as latin-1: {repr(bad_decode)}')
# No error raised, but the output is garbage — dangerous!

In [ ]:
# Detecting encoding with chardet
# pip install chardet (already available in most environments)
try:
    import chardet

    # Simulate receiving a file with unknown encoding
    mystery_bytes = 'Bonjour le caf\u00e9'.encode('latin-1')

    result = chardet.detect(mystery_bytes)
    print(f'chardet detection: {result}')
    # Often correct, but confidence can be low on short samples
    # Always validate the result before trusting it
except ImportError:
    print('chardet not installed. Install with: pip install chardet')
    print('Showing encoding detection with raw heuristics instead.')

# Manual check: try UTF-8 first, fall back to latin-1
def decode_best_effort(raw_bytes: bytes) -> str:
    '''Try UTF-8 first; fall back to latin-1 (which never fails).'''
    try:
        return raw_bytes.decode('utf-8')
    except UnicodeDecodeError:
        return raw_bytes.decode('latin-1')

mystery = 'prix: 12\u20ac'.encode('utf-8')  # euro sign
print(decode_best_effort(mystery))

In [ ]:
# Byte Order Mark (BOM) — a hidden character at the start of some files
# Excel often adds a UTF-8 BOM (EF BB BF) when saving as CSV
# This breaks field name parsing because the first column header gets a hidden prefix

bom_bytes = b'\xef\xbb\xbf' + b'order_id,product,total\n1001,Laptop,1200\n'

# Reading without BOM handling — first column name is wrong
with io.TextIOWrapper(io.BytesIO(bom_bytes), encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f'Column names: {list(row.keys())}')
        print(f'order_id value: {row.get("order_id", "NOT FOUND")}')
        break

print()

# Reading with utf-8-sig: Python strips the BOM automatically
with io.TextIOWrapper(io.BytesIO(bom_bytes), encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f'Column names: {list(row.keys())}')
        print(f'order_id value: {row.get("order_id", "NOT FOUND")}')
        break
# Use utf-8-sig when reading files exported from Excel

In [ ]:
# Encoding error handlers — what to do when a byte cannot be decoded
# errors='strict'      — raise UnicodeDecodeError (default)
# errors='replace'     — insert the replacement character U+FFFD for bad bytes
# errors='ignore'      — silently drop bad bytes
# errors='surrogateescape' — preserve bad bytes as surrogate characters, re-encode them later

bad_bytes = b'Order ID\xff: 1001'  # \xff is invalid in UTF-8

with io.TextIOWrapper(io.BytesIO(bad_bytes), encoding='utf-8', errors='replace') as f:
    content = f.read()
    print(f'replace: {repr(content)}')
    # The \xff becomes U+FFFD (the diamond question mark)

with io.TextIOWrapper(io.BytesIO(bad_bytes), encoding='utf-8', errors='ignore') as f:
    content = f.read()
    print(f'ignore:  {repr(content)}')
    # The \xff byte is silently dropped

# For data engineering: use 'replace' when you want to see where bad bytes are
# Use 'ignore' only when you are certain the bad bytes are not data you need

---

## 2.3 The Context Manager Protocol

The `with` statement is not magic. It calls two special methods on the object:

1. `__enter__()` — called when entering the `with` block. Its return value is bound to the `as` name.
2. `__exit__(exc_type, exc_val, exc_tb)` — called when leaving the block, whether by normal exit **or by an exception**.

If `__exit__` returns `True`, any exception is suppressed. If it returns `False` or `None`, the exception propagates.

This guarantees cleanup. A file opened in a `with` block is closed even if an exception is raised halfway through reading. A database connection is released even if the query fails.

In [ ]:
# Implement __enter__ and __exit__ from scratch to see how with works

class ManagedFile:
    def __init__(self, path, mode='r', encoding='utf-8'):
        self.path = path
        self.mode = mode
        self.encoding = encoding
        self.file = None

    def __enter__(self):
        print(f'__enter__: opening {self.path}')
        self.file = open(self.path, self.mode, encoding=self.encoding)
        return self.file    # this is what the `as f` name receives

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f'__exit__: closing {self.path}')
        if self.file:
            self.file.close()
        if exc_type is not None:
            print(f'__exit__: exception was {exc_type.__name__}: {exc_val}')
        return False  # do not suppress exceptions

with ManagedFile(RAW / 'sales_q1.csv') as f:
    first_line = f.readline()
    print(f'first line: {first_line.strip()}')
# __exit__ is called here automatically

In [ ]:
# __exit__ is called even when an exception occurs
print('=== with exception ===')
try:
    with ManagedFile(RAW / 'sales_q1.csv') as f:
        _ = f.readline()
        raise ValueError('simulated processing error')
except ValueError as e:
    print(f'caught: {e}')
# You will see __exit__ is called before the exception propagates

In [ ]:
# contextlib.contextmanager: write a context manager with a generator function
# yield once: everything before yield is __enter__, after yield is __exit__

from contextlib import contextmanager

@contextmanager
def timed_step(name: str):
    '''Context manager that times a block and prints the duration.'''
    import time
    print(f'[START] {name}')
    start = time.perf_counter()
    try:
        yield  # the with block runs here
    finally:
        elapsed = time.perf_counter() - start
        print(f'[END]   {name}: {elapsed * 1000:.1f} ms')

# Use it to time file operations
with timed_step('reading sales_q1.csv'):
    with open(RAW / 'sales_q1.csv', 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'  read {len(rows)} rows')

with timed_step('reading all quarterly files'):
    all_rows = []
    for path in sorted(RAW.glob('*.csv')):
        with open(path, 'r', encoding='utf-8') as f:
            all_rows.extend(csv.DictReader(f))
    print(f'  total rows: {len(all_rows)}')

In [ ]:
# Multiple context managers in one with statement
# Equivalent to nested with blocks, but cleaner
q1_path = RAW / 'sales_q1.csv'
q2_path = RAW / 'sales_q2.csv'

with open(q1_path, 'r', encoding='utf-8') as f1, \
     open(q2_path, 'r', encoding='utf-8') as f2:
    r1 = list(csv.DictReader(f1))
    r2 = list(csv.DictReader(f2))
    print(f'Q1: {len(r1)} rows, Q2: {len(r2)} rows')

---

## 2.4 Streaming Large Files

Loading a large file entirely into a list before processing is the most common mistake in data engineering. A 1 GB CSV file becomes several gigabytes in memory once Python converts it to dicts. A 100 GB file crashes the process.

The solution is to **process rows as a stream**: read one row, process it, write it, discard it, read the next. At any point, only one row is in memory.

Python files are iterators — `for line in f` reads one line at a time from the OS buffer without loading the whole file. `csv.DictReader` wraps this iterator, so `for row in reader` is also streaming.

In [ ]:
# Measure memory: loading all rows vs streaming
tracemalloc.start()

# Method A: load all rows into a list
tracemalloc.reset_peak()
with open(RAW / 'sales_q1.csv', 'r', encoding='utf-8') as f:
    rows_in_memory = list(csv.DictReader(f))
_, peak_load = tracemalloc.get_traced_memory()

# Method B: stream rows and count without storing
tracemalloc.reset_peak()
count = 0
with open(RAW / 'sales_q1.csv', 'r', encoding='utf-8') as f:
    for _row in csv.DictReader(f):
        count += 1
_, peak_stream = tracemalloc.get_traced_memory()

tracemalloc.stop()

print(f'Load all:  peak {peak_load / 1024:.1f} KB   ({len(rows_in_memory)} dicts in memory)')
print(f'Stream:    peak {peak_stream / 1024:.1f} KB   ({count} rows processed, none stored)')
print()
print('With a 1 GB file, loading would use ~3-5 GB RAM. Streaming would use ~1 KB.')

In [ ]:
# Generator pipeline: read -> clean -> write without materialising anything

def iter_csv(path, encoding='utf-8'):
    '''Yield one dict per row from a CSV file.'''
    with open(path, 'r', newline='', encoding=encoding) as f:
        yield from csv.DictReader(f)
    # yield from delegates to the iterator
    # The file stays open only while this generator is being iterated

def clean_stream(rows):
    '''Yield cleaned rows. Drop rows with quantity <= 0.'''
    for row in rows:
        row['category'] = row['category'].strip().title()
        row['product'] = row['product'].strip()
        try:
            qty = float(row['quantity'])
        except (ValueError, TypeError):
            continue  # drop unparseable
        if qty <= 0:
            continue  # drop invalid
        row['quantity'] = int(qty)
        yield row

def write_csv_stream(rows, path, fieldnames):
    '''Write a stream of dicts to CSV. Returns row count.'''
    count = 0
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
            count += 1
    return count

# Wire the pipeline
FIELDNAMES = ['order_id', 'date', 'customer_name', 'product', 'category',
               'quantity', 'unit_price', 'total', 'region', 'sales_rep']
out_path = Path('../data/clean/sales_q1_streamed.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)

n = write_csv_stream(
    clean_stream(iter_csv(RAW / 'sales_q1.csv')),
    out_path,
    FIELDNAMES
)
print(f'Written: {n} rows to {out_path}')

In [ ]:
# io.StringIO: in-memory file for testing without touching disk
# Useful in tests where you want to supply CSV data directly

csv_text = '''order_id,product,category,quantity,total
1001,Laptop,Electronics,2,2400
1002,Mouse,Electronics,-1,35
1003,Jacket,Clothing,1,120
'''

buffer = io.StringIO(csv_text)
reader = csv.DictReader(buffer)

for row in clean_stream(reader):
    print(row)
# row with quantity=-1 is filtered out

---

## 2.5 Exception Handling for I/O

File I/O fails in predictable ways. Knowing the exception hierarchy lets you catch exactly what you mean to catch — neither too broad nor too narrow.

```
BaseException
  Exception
    OSError  (also aliased as IOError, EnvironmentError)
      FileNotFoundError     # file does not exist
      PermissionError       # no read/write access
      IsADirectoryError     # opened a directory as a file
      FileExistsError       # tried to create a file that already exists
    ValueError              # encoding/format errors from csv/json
    UnicodeDecodeError      # encoding mismatch
```

Never catch `Exception` or bare `except:` in a pipeline. Catching too broadly swallows bugs silently. Always catch the most specific exception that represents the failure mode you are handling.

In [ ]:
# Prove the exception hierarchy with isinstance
err = FileNotFoundError('test')
print(f'FileNotFoundError is OSError:    {isinstance(err, OSError)}')
print(f'FileNotFoundError is Exception:  {isinstance(err, Exception)}')
print(f'FileNotFoundError is ValueError: {isinstance(err, ValueError)}')

# Catching at the right level
def read_csv_safe(path: Path) -> list:
    '''Read a CSV; return empty list and print a specific error on failure.'''
    try:
        with open(path, 'r', newline='', encoding='utf-8') as f:
            return list(csv.DictReader(f))
    except FileNotFoundError:
        print(f'ERROR: file not found: {path}')
        return []
    except PermissionError:
        print(f'ERROR: permission denied: {path}')
        return []
    except UnicodeDecodeError as e:
        print(f'ERROR: encoding problem in {path}: {e}')
        return []

print(read_csv_safe(Path('does_not_exist.csv')))
print(len(read_csv_safe(RAW / 'sales_q1.csv')))

In [ ]:
# Exception chaining: raise ... from ...
# When one exception causes another, preserve the original as context

def load_config(path: Path) -> dict:
    '''Load a JSON config file. Raises ValueError with a useful message on failure.'''
    import json
    try:
        text = path.read_text(encoding='utf-8')
    except FileNotFoundError as e:
        raise FileNotFoundError(f'Config file not found: {path}') from e
        # The original FileNotFoundError is preserved as __cause__
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f'Config file is not valid JSON: {path}') from e

try:
    load_config(Path('missing_config.json'))
except FileNotFoundError as e:
    print(f'Caught: {e}')
    print(f'Caused by: {e.__cause__}')

In [ ]:
# try/except/else/finally — all four clauses
# else: runs only if NO exception occurred in try
# finally: runs ALWAYS, exception or not

def process_file(path: Path):
    print(f'Processing {path.name}')
    rows = []
    try:
        with open(path, 'r', newline='', encoding='utf-8') as f:
            rows = list(csv.DictReader(f))
    except FileNotFoundError:
        print(f'  SKIP: not found')
        return []
    except UnicodeDecodeError as e:
        print(f'  SKIP: encoding error: {e}')
        return []
    else:
        print(f'  OK: {len(rows)} rows')
        # else only runs when try succeeded — clean separation of success logic
    finally:
        print(f'  DONE (always printed)')
        # finally always runs — even on success, even on exception, even on return
    return rows

result = process_file(RAW / 'sales_q1.csv')
result = process_file(Path('nonexistent.csv'))

---

## 2.6 pathlib in Depth

`pathlib.Path` represents a file system path as an object rather than a string. It has methods for every common operation: join, check existence, list directories, get size, read/write text, create directories, move files.

Key advantage over `os.path`: operator overloading. `base / 'subdir' / 'file.csv'` is readable and works on every OS.

In [ ]:
# Path anatomy
p = Path('/Users/student/lessons/week3/data/raw/sales_q1.csv')

print(f'name:    {p.name}')        # sales_q1.csv
print(f'stem:    {p.stem}')        # sales_q1
print(f'suffix:  {p.suffix}')      # .csv
print(f'parent:  {p.parent}')      # .../data/raw
print(f'parts:   {p.parts}')       # tuple of all parts

# Building paths
base = Path('../data')
raw_dir  = base / 'raw'
clean_dir = base / 'clean'
q1_file  = raw_dir / 'sales_q1.csv'
print(f'\nbuilt: {q1_file}')

# Path is not relative until you call .resolve()
print(f'relative: {q1_file}')
print(f'absolute: {q1_file.resolve()}')

In [ ]:
# File system inspection
raw_path = RAW

print(f'exists:      {raw_path.exists()}')
print(f'is_dir:      {raw_path.is_dir()}')
print(f'is_file:     {raw_path.is_file()}')

# List all CSV files
csv_files = sorted(raw_path.glob('*.csv'))
for f in csv_files:
    stat = f.stat()
    size_kb = stat.st_size / 1024
    import datetime
    mtime = datetime.datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M')
    print(f'  {f.name:20} {size_kb:6.1f} KB  modified {mtime}')

In [ ]:
# Atomic writes: the correct pattern for writing files in a pipeline
# Problem: if the process crashes mid-write, you get a half-written file
# that looks complete but is corrupt
# Solution: write to a temp file, then rename (atomic on most file systems)

def atomic_write_csv(rows: list, dest: Path, fieldnames: list) -> None:
    '''Write rows to a temporary file and rename to dest atomically.'''
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix('.tmp')
    try:
        with open(tmp, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(rows)
        tmp.rename(dest)    # atomic on POSIX systems: either succeeds or fails completely
        print(f'Written: {dest}')
    except Exception:
        if tmp.exists():
            tmp.unlink()    # clean up the partial file
        raise               # re-raise so the caller knows it failed

FIELDS = ['order_id', 'product', 'total']
sample = [{'order_id': '1001', 'product': 'Laptop', 'total': '1200'}]
atomic_write_csv(sample, Path('../data/clean/test_atomic.csv'), FIELDS)

# Verify
written = Path('../data/clean/test_atomic.csv')
print(written.read_text())

In [ ]:
# Recursive glob: ** matches any number of directories
lesson_dir = Path('..')
all_csvs = sorted(lesson_dir.glob('**/*.csv'))
print(f'All CSV files under week3:')
for csv_path in all_csvs:
    relative = csv_path.relative_to(lesson_dir)
    print(f'  {relative}')

# Cleaning up the test file
Path('../data/clean/test_atomic.csv').unlink(missing_ok=True)
Path('../data/clean/sales_q1_streamed.csv').unlink(missing_ok=True)

---

## Summary

| Concept | What to remember |
|---------|------------------|
| File descriptors | Every open file consumes a resource. `with` guarantees release. Running out of file descriptors crashes a pipeline silently. |
| Text vs binary mode | Open in binary (`rb`) to inspect raw bytes or detect encoding. Open in text (`r`) with an explicit `encoding=` for all CSV and text work. |
| Encoding | Always specify `encoding='utf-8'`. Use `utf-8-sig` for Excel CSV exports. Latin-1 never raises an error but silently corrupts non-ASCII data when decoded. |
| Context manager protocol | `__enter__` and `__exit__`. Any class can be a context manager. Use `@contextmanager` for lightweight ones. `__exit__` runs even on exceptions. |
| Streaming | `for row in csv.DictReader(f)` is streaming — one dict at a time. Never call `list()` on it unless you need the whole dataset in memory. |
| Generator pipelines | Chain generator functions to build streaming pipelines. Memory usage is O(1) in the number of rows. |
| Exception hierarchy | `FileNotFoundError` and `PermissionError` are both `OSError`. Catch the most specific class that matches the failure mode. Always chain exceptions with `raise X from Y`. |
| Atomic writes | Write to `.tmp`, then rename. A failed write leaves the original file intact. |